# Baselines + MLflow
## Previsão de Churn - Telco Customer

**Objetivo:** treinar modelos baseline, avaliar métricas e registrar experimentos no MLflow.

**Modelos:**
- DummyClassifier (referência mínima)
- Logistic Regression (baseline interpretável)
- Random Forest (baseline robusto)

**Métricas:** Accuracy, Precision, Recall, F1-score, ROC-AUC

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import mlflow
import mlflow.sklearn

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Semente aleatoria para reprodutibilidade
SEED = 42
np.random.seed(SEED)

print('Bibliotecas carregadas com sucesso!')
print(f'Seed: {SEED}')

## 1. Carregar Dados Processados

In [ ]:
# Carregar dados limpos da EDA
df = pd.read_csv('../data/processed/telco_churn_clean.csv')

print(f'Dataset carregado!')
print(f'Shape: {df.shape[0]} linhas x {df.shape[1]} colunas')
print()
df.head()

## 2. Preparação dos Dados
### 2.1 Separar Features e Target

In [ ]:
# Definir target
TARGET = 'Churn'

# Converter Churn para numerico: Yes=1, No=0
df[TARGET] = df[TARGET].map({'Yes': 1, 'No': 0})

# Separar features (X) e target (y)
X = df.drop(TARGET, axis=1)
y = df[TARGET]

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print()
print('Distribuicao do target:')
print(y.value_counts())
print()
print('Proporcao:')
print(y.value_counts(normalize=True).round(4) * 100)

### 2.2 Encoding das Variáveis Categóricas

In [ ]:
# One-Hot Encoding das variaveis categoricas
X = pd.get_dummies(X, drop_first=True)

print(f'Shape apos encoding: {X.shape}')
print(f'\nPrimeiras 10 colunas: {list(X.columns[:10])}')
print(f'Total de features: {X.shape[1]}')

### 2.3 Split Treino/Teste

In [ ]:
# Dividir em treino e teste (80/20) com estratificacao
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.0f}%)')
print()
print(f'Proporcao Churn no treino: {y_train.mean():.4f} ({y_train.mean()*100:.1f}%)')
print(f'Proporcao Churn no teste:  {y_test.mean():.4f} ({y_test.mean()*100:.1f}%)')
print()
print('Estratificacao OK!' if abs(y_train.mean() - y_test.mean()) < 0.01 else 'ATENCAO: proporcoes diferentes!')

### 2.4 Escalonamento

In [ ]:
# Escalonar features (importante para Logistic Regression e MLP futura)
scaler = StandardScaler()

# Fit APENAS no treino (evitar data leakage)
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Transform no teste
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print('Escalonamento aplicado!')
print(f'Media treino (primeiras 3 features): {X_train_scaled.iloc[:, :3].mean().values.round(4)}')
print(f'Std treino (primeiras 3 features):   {X_train_scaled.iloc[:, :3].std().values.round(4)}')

## 3. Treinamento dos Baselines

Vamos treinar 3 modelos e comparar suas métricas.

### 3.1 Função auxiliar de avaliação

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Treina o modelo, faz predicoes e calcula metricas.
    Retorna dicionario com resultados.
    """
    # Treinar
    model.fit(X_train, y_train)
    
    # Prever
    y_pred = model.predict(X_test)
    
    # Probabilidades (para ROC-AUC)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = 0.5  # sem probabilidade
    
    # Metricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Resultados
    print(f'=== {model_name} ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print(f'ROC-AUC:   {roc_auc:.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    
    # Matriz de confusao
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')
    ax.set_title(f'Matriz de Confusao - {model_name}')
    plt.tight_layout()
    plt.show()
    
    return {
        'model_name': model_name,
        'accuracy': round(acc, 4),
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f1_score': round(f1, 4),
        'roc_auc': round(roc_auc, 4),
        'model': model
    }

print('Funcao de avaliacao criada!')

### 3.2 Modelo 1 — DummyClassifier

O DummyClassifier é um modelo que faz previsões **sem aprender padrões**. Serve como **referência mínima**: qualquer modelo real deve ser melhor que ele.

In [ ]:
# Treinar DummyClassifier
dummy = DummyClassifier(strategy='stratified', random_state=SEED)
result_dummy = evaluate_model(dummy, X_train_scaled, X_test_scaled, y_train, y_test, 'DummyClassifier')

### 3.3 Modelo 2 — Logistic Regression

A Regressão Logística é um modelo linear simples e interpretável. É o **baseline inteligente** clássico para problemas de classificação.

In [ ]:
# Treinar Logistic Regression
lr = LogisticRegression(random_state=SEED, max_iter=1000)
result_lr = evaluate_model(lr, X_train_scaled, X_test_scaled, y_train, y_test, 'LogisticRegression')

### 3.4 Modelo 3 — Random Forest

O Random Forest é um ensemble de árvores de decisão. Geralmente tem boa performance e serve como **baseline robusto**.

In [ ]:
# Treinar Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
result_rf = evaluate_model(rf, X_train_scaled, X_test_scaled, y_train, y_test, 'RandomForest')

## 4. Comparação dos Modelos

In [ ]:
# Criar tabela comparativa
results = [result_dummy, result_lr, result_rf]

# Remover coluna 'model' para exibicao
results_display = [{k: v for k, v in r.items() if k != 'model'} for r in results]
df_results = pd.DataFrame(results_display).set_index('model_name')

print('=== TABELA COMPARATIVA ===')
print()
print(df_results.to_string())
print()

# Grafico comparativo
metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))

bars1 = ax.bar(x - width, df_results.loc['DummyClassifier', metrics], width, label='DummyClassifier', color='#95a5a6')
bars2 = ax.bar(x, df_results.loc['LogisticRegression', metrics], width, label='LogisticRegression', color='#3498db')
bars3 = ax.bar(x + width, df_results.loc['RandomForest', metrics], width, label='RandomForest', color='#2ecc71')

ax.set_ylabel('Score')
ax.set_title('Comparacao dos Modelos Baseline', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics])
ax.legend()
ax.set_ylim(0, 1.1)

# Adicionar valores nas barras
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Validação Cruzada Estratificada

Para garantir que os resultados não dependem de uma única divisão dos dados, vamos usar validação cruzada estratificada com 5 folds.

In [ ]:
# Validacao cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Logistic Regression
lr_cv = LogisticRegression(random_state=SEED, max_iter=1000)
scores_lr = cross_val_score(lr_cv, X_train_scaled, y_train, cv=cv, scoring='roc_auc')

# Random Forest
rf_cv = RandomForestClassifier(n_estimators=100, random_state=SEED)
scores_rf = cross_val_score(rf_cv, X_train_scaled, y_train, cv=cv, scoring='roc_auc')

print('=== Validacao Cruzada (5-Fold) - ROC-AUC ===')
print()
print(f'Logistic Regression: {scores_lr.mean():.4f} (+/- {scores_lr.std():.4f})')
print(f'  Folds: {scores_lr.round(4)}')
print()
print(f'Random Forest:       {scores_rf.mean():.4f} (+/- {scores_rf.std():.4f})')
print(f'  Folds: {scores_rf.round(4)}')
print()

melhor = 'Logistic Regression' if scores_lr.mean() > scores_rf.mean() else 'Random Forest'
print(f'Melhor baseline por CV: {melhor}')

## 6. Registro no MLflow

Vamos registrar todos os experimentos no MLflow para rastreabilidade.

In [ ]:
# Configurar MLflow
mlflow.set_experiment('churn-baselines')

# Registrar cada modelo
for result in results:
    model_name = result['model_name']
    model = result['model']
    
    with mlflow.start_run(run_name=model_name):
        # Log parametros
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('random_state', SEED)
        mlflow.log_param('test_size', 0.2)
        mlflow.log_param('scaling', 'StandardScaler')
        
        # Log metricas
        mlflow.log_metric('accuracy', result['accuracy'])
        mlflow.log_metric('precision', result['precision'])
        mlflow.log_metric('recall', result['recall'])
        mlflow.log_metric('f1_score', result['f1_score'])
        mlflow.log_metric('roc_auc', result['roc_auc'])
        
        # Log modelo
        mlflow.sklearn.log_model(model, model_name)
        
        print(f'[OK] {model_name} registrado no MLflow')

print()
print('Todos os experimentos registrados!')
print()
print('Para visualizar o MLflow UI, rode no terminal:')
print('  mlflow ui')
print('Depois acesse: http://localhost:5000')

## 7. Salvar Melhor Baseline

In [ ]:
import os

# Identificar melhor baseline por ROC-AUC
best = max(results, key=lambda x: x['roc_auc'])
best_name = best['model_name']
best_model = best['model']

print(f'Melhor baseline: {best_name} (ROC-AUC: {best["roc_auc"]})')
print()

# Salvar modelo
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_baseline.pkl')
print(f'[OK] Modelo salvo em: models/best_baseline.pkl')

# Salvar scaler
joblib.dump(scaler, '../models/scaler.pkl')
print(f'[OK] Scaler salvo em: models/scaler.pkl')

# Salvar tabela de resultados
df_results.to_csv('../models/baseline_results.csv')
print(f'[OK] Resultados salvos em: models/baseline_results.csv')

## 8. Conclusões dos Baselines

### Resultados
- Os 3 modelos foram treinados e avaliados com sucesso
- A validação cruzada estratificada confirmou a estabilidade dos resultados
- Todos os experimentos foram registrados no MLflow

### Análise
- O **DummyClassifier** serve como referência mínima — qualquer modelo real deve superá-lo
- A **Regressão Logística** já mostra que o problema é modelável e os dados contêm sinais preditivos
- O **Random Forest** geralmente oferece melhor performance entre os baselines
- Todos os baselines serão comparados com a **MLP (PyTorch)** na próxima fase

### Trade-off de Custo
- **Falso Negativo** (cliente cancela mas modelo não detecta): alto custo — perda do cliente
- **Falso Positivo** (modelo alerta mas cliente não cancelaria): custo moderado — ação de retenção desnecessária
- Para churn, **Recall é especialmente importante** para não perder clientes

### Próximos Passos
1. **Fase 3** — Treinar MLP em PyTorch
2. Comparar MLP com esses baselines
3. Analisar trade-off de custo detalhadamente
4. Feature Engineering adicional se necessário